In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

print("Gold layer notebook initialized successfully.")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 3, Finished, Available, Finished, False)

Gold layer notebook initialized successfully.


In [2]:
silver_customers = spark.table("silver_customers")
silver_drivers = spark.table("silver_drivers")
silver_trucks = spark.table("silver_trucks")
silver_trailers = spark.table("silver_trailers")
silver_facilities = spark.table("silver_facilities")
silver_routes = spark.table("silver_routes")
silver_loads = spark.table("silver_loads")
silver_trips = spark.table("silver_trips")
silver_fuel_purchases = spark.table("silver_fuel_purchases")
silver_maintenance = spark.table("silver_maintenance")
silver_delivery_events = spark.table("silver_delivery_events")
silver_safety_incidents = spark.table("silver_safety_incidents")

print("All Silver tables loaded successfully.")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 4, Finished, Available, Finished, False)

All Silver tables loaded successfully.


In [3]:
silver_counts = {
    "silver_customers": silver_customers.count(),
    "silver_drivers": silver_drivers.count(),
    "silver_trucks": silver_trucks.count(),
    "silver_trailers": silver_trailers.count(),
    "silver_facilities": silver_facilities.count(),
    "silver_routes": silver_routes.count(),
    "silver_loads": silver_loads.count(),
    "silver_trips": silver_trips.count(),
    "silver_fuel_purchases": silver_fuel_purchases.count(),
    "silver_maintenance": silver_maintenance.count(),
    "silver_delivery_events": silver_delivery_events.count(),
    "silver_safety_incidents": silver_safety_incidents.count()
}

for table_name, row_count in silver_counts.items():
    print(f"{table_name}: {row_count:,}")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 5, Finished, Available, Finished, False)

silver_customers: 200
silver_drivers: 150
silver_trucks: 120
silver_trailers: 180
silver_facilities: 50
silver_routes: 58
silver_loads: 85,410
silver_trips: 85,410
silver_fuel_purchases: 196,442
silver_maintenance: 2,920
silver_delivery_events: 170,820
silver_safety_incidents: 170


In [4]:
date_ranges = [
    silver_customers.select(
        F.min("contract_start_date").alias("min_date"),
        F.max("contract_start_date").alias("max_date")
    ),
    silver_drivers.select(
        F.min("hire_date").alias("min_date"),
        F.max("hire_date").alias("max_date")
    ),
    silver_loads.select(
        F.min("load_date").alias("min_date"),
        F.max("load_date").alias("max_date")
    ),
    silver_trips.select(
        F.min("dispatch_date").alias("min_date"),
        F.max("dispatch_date").alias("max_date")
    ),
    silver_fuel_purchases.select(
        F.min("purchase_date").alias("min_date"),
        F.max("purchase_date").alias("max_date")
    ),
    silver_maintenance.select(
        F.min("maintenance_date").alias("min_date"),
        F.max("maintenance_date").alias("max_date")
    ),
    silver_safety_incidents.select(
        F.min("incident_date").alias("min_date"),
        F.max("incident_date").alias("max_date")
    )
]

combined_dates = date_ranges[0]

for df in date_ranges[1:]:
    combined_dates = combined_dates.union(df)

date_range = combined_dates.select(
    F.min("min_date").alias("min_date"),
    F.max("max_date").alias("max_date")
).collect()[0]

min_date = date_range["min_date"]
max_date = date_range["max_date"]

print(f"Minimum date: {min_date}")
print(f"Maximum date: {max_date}")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 6, Finished, Available, Finished, False)

Minimum date: 2012-01-28
Maximum date: 2025-01-02


In [5]:
date_df = (
    spark.sql("""
        SELECT explode(
            sequence(
                to_date('2012-01-28'),
                to_date('2025-01-02'),
                interval 1 day
            )
        ) AS full_date
    """)
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("week_of_year", F.weekofyear("full_date"))
    .withColumn("day_of_month", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn(
        "is_weekend",
        F.when(F.dayofweek("full_date").isin([1, 7]), True).otherwise(False)
    )
    .select(
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "week_of_year",
        "day_of_month",
        "day_of_week",
        "day_name",
        "is_weekend"
    )
)

print(f"Date dimension rows: {date_df.count():,}")

display(
    date_df.orderBy("full_date")
)

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 7, Finished, Available, Finished, False)

Date dimension rows: 4,724


SynapseWidget(Synapse.DataFrame, 3669a798-c9dc-4997-89cb-a35dedfbb791)

In [6]:
date_count = date_df.count()

null_dates = date_df.filter(
    F.col("full_date").isNull()
).count()

duplicate_keys = (
    date_df
    .groupBy("date_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Date dimension records: {date_count:,}")
print(f"NULL dates: {null_dates}")
print(f"Duplicate date keys: {duplicate_keys}")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 8, Finished, Available, Finished, False)

Date dimension records: 4,724
NULL dates: 0
Duplicate date keys: 0


In [7]:
date_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_date")

print("gold_dim_date created successfully.")
print(
    f"Gold date dimension records: "
    f"{spark.table('gold_dim_date').count():,}"
)

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 9, Finished, Available, Finished, False)

gold_dim_date created successfully.
Gold date dimension records: 4,724


In [8]:
customer_dim = (
    silver_customers
    .select(
        "customer_id",
        "customer_name",
        "customer_type",
        "credit_terms_days",
        "primary_freight_type",
        "account_status",
        "contract_start_date",
        "customer_tenure_years"
    )
    .dropDuplicates(["customer_id"])
    .withColumn(
        "customer_key",
        F.row_number().over(
            Window.orderBy("customer_id")
        )
    )
    .select(
        "customer_key",
        "customer_id",
        "customer_name",
        "customer_type",
        "credit_terms_days",
        "primary_freight_type",
        "account_status",
        "contract_start_date",
        "customer_tenure_years"
    )
)

print(f"Customer dimension records: {customer_dim.count():,}")

display(
    customer_dim.orderBy("customer_key")
)

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 10, Finished, Available, Finished, False)

Customer dimension records: 200


SynapseWidget(Synapse.DataFrame, ef2aaba6-91f0-483d-96a8-f303ae5e3fb6)

In [9]:
customer_count = customer_dim.count()

null_customer_ids = customer_dim.filter(
    F.col("customer_id").isNull()
).count()

duplicate_customer_ids = (
    customer_dim
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_customer_keys = (
    customer_dim
    .groupBy("customer_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Customer dimension records: {customer_count:,}")
print(f"NULL customer IDs: {null_customer_ids}")
print(f"Duplicate customer IDs: {duplicate_customer_ids}")
print(f"Duplicate customer keys: {duplicate_customer_keys}")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 11, Finished, Available, Finished, False)

Customer dimension records: 200
NULL customer IDs: 0
Duplicate customer IDs: 0
Duplicate customer keys: 0


In [10]:
customer_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_customer")

print("gold_dim_customer created successfully.")
print(
    f"Gold customer dimension records: "
    f"{spark.table('gold_dim_customer').count():,}"
)

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 12, Finished, Available, Finished, False)

gold_dim_customer created successfully.
Gold customer dimension records: 200


In [11]:
driver_dim = (
    silver_drivers
    .select(
        "driver_id",
        "first_name",
        "last_name",
        "hire_date",
        "termination_date",
        "license_number",
        "license_state",
        "date_of_birth",
        "home_terminal",
        "employment_status",
        "cdl_class",
        "years_experience",
        "driver_age_years"
    )
    .dropDuplicates(["driver_id"])
    .withColumn(
        "driver_key",
        F.row_number().over(
            Window.orderBy("driver_id")
        )
    )
    .select(
        "driver_key",
        "driver_id",
        "first_name",
        "last_name",
        "hire_date",
        "termination_date",
        "license_number",
        "license_state",
        "date_of_birth",
        "home_terminal",
        "employment_status",
        "cdl_class",
        "years_experience",
        "driver_age_years"
    )
)

print(f"Driver dimension records: {driver_dim.count():,}")

display(
    driver_dim.orderBy("driver_key")
)

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 13, Finished, Available, Finished, False)

Driver dimension records: 150


SynapseWidget(Synapse.DataFrame, 1f967056-820c-46f5-a3ef-0a944085e9d8)

In [12]:
driver_count = driver_dim.count()

null_driver_ids = driver_dim.filter(
    F.col("driver_id").isNull()
).count()

duplicate_driver_ids = (
    driver_dim
    .groupBy("driver_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_driver_keys = (
    driver_dim
    .groupBy("driver_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Driver dimension records: {driver_count:,}")
print(f"NULL driver IDs: {null_driver_ids}")
print(f"Duplicate driver IDs: {duplicate_driver_ids}")
print(f"Duplicate driver keys: {duplicate_driver_keys}")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 14, Finished, Available, Finished, False)

Driver dimension records: 150
NULL driver IDs: 0
Duplicate driver IDs: 0
Duplicate driver keys: 0


In [13]:
driver_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_driver")

print("gold_dim_driver created successfully.")
print(
    f"Gold driver dimension records: "
    f"{spark.table('gold_dim_driver').count():,}"
)

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 15, Finished, Available, Finished, False)

gold_dim_driver created successfully.
Gold driver dimension records: 150


In [14]:
truck_dim = (
    silver_trucks
    .select(
        "truck_id",
        "unit_number",
        "make",
        "model_year",
        "vin",
        "acquisition_date",
        "acquisition_mileage",
        "fuel_type",
        "tank_capacity_gallons",
        "status",
        "home_terminal"
    )
    .dropDuplicates(["truck_id"])
    .withColumn(
        "truck_key",
        F.row_number().over(Window.orderBy("truck_id"))
    )
    .select(
        "truck_key",
        "truck_id",
        "unit_number",
        "make",
        "model_year",
        "vin",
        "acquisition_date",
        "acquisition_mileage",
        "fuel_type",
        "tank_capacity_gallons",
        "status",
        "home_terminal"
    )
)

print(f"Truck dimension records: {truck_dim.count():,}")

truck_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_truck")

print("gold_dim_truck created successfully.")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 16, Finished, Available, Finished, False)

Truck dimension records: 120
gold_dim_truck created successfully.


In [15]:
trailer_dim = (
    silver_trailers
    .select(
        "trailer_id",
        "trailer_number",
        "trailer_type",
        "length_feet",
        "model_year",
        "vin",
        "acquisition_date",
        "status",
        "current_location"
    )
    .dropDuplicates(["trailer_id"])
    .withColumn(
        "trailer_key",
        F.row_number().over(Window.orderBy("trailer_id"))
    )
    .select(
        "trailer_key",
        "trailer_id",
        "trailer_number",
        "trailer_type",
        "length_feet",
        "model_year",
        "vin",
        "acquisition_date",
        "status",
        "current_location"
    )
)

print(f"Trailer dimension records: {trailer_dim.count():,}")

trailer_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_trailer")

print("gold_dim_trailer created successfully.")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 17, Finished, Available, Finished, False)

Trailer dimension records: 180
gold_dim_trailer created successfully.


In [16]:
facility_dim = (
    silver_facilities
    .select(
        "facility_id",
        "facility_name",
        "facility_type",
        "city",
        "state",
        "latitude",
        "longitude",
        "dock_doors",
        "operating_hours"
    )
    .dropDuplicates(["facility_id"])
    .withColumn(
        "facility_key",
        F.row_number().over(Window.orderBy("facility_id"))
    )
    .select(
        "facility_key",
        "facility_id",
        "facility_name",
        "facility_type",
        "city",
        "state",
        "latitude",
        "longitude",
        "dock_doors",
        "operating_hours"
    )
)

print(f"Facility dimension records: {facility_dim.count():,}")

facility_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_facility")

print("gold_dim_facility created successfully.")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 18, Finished, Available, Finished, False)

Facility dimension records: 50
gold_dim_facility created successfully.


In [17]:
route_dim = (
    silver_routes
    .select(
        "route_id",
        "origin_city",
        "origin_state",
        "destination_city",
        "destination_state",
        "typical_distance_miles",
        "base_rate_per_mile",
        "fuel_surcharge_rate",
        "typical_transit_days"
    )
    .dropDuplicates(["route_id"])
    .withColumn(
        "route_key",
        F.row_number().over(Window.orderBy("route_id"))
    )
    .select(
        "route_key",
        "route_id",
        "origin_city",
        "origin_state",
        "destination_city",
        "destination_state",
        "typical_distance_miles",
        "base_rate_per_mile",
        "fuel_surcharge_rate",
        "typical_transit_days"
    )
)

print(f"Route dimension records: {route_dim.count():,}")

route_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_route")

print("gold_dim_route created successfully.")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 19, Finished, Available, Finished, False)

Route dimension records: 58
gold_dim_route created successfully.


In [18]:
customer_lookup = spark.table("gold_dim_customer").select(
    "customer_key",
    "customer_id"
)

route_lookup = spark.table("gold_dim_route").select(
    "route_key",
    "route_id"
)

date_lookup = spark.table("gold_dim_date").select(
    "date_key",
    "full_date"
)

load_fact = (
    silver_loads
    .join(
        customer_lookup,
        "customer_id",
        "left"
    )
    .join(
        route_lookup,
        "route_id",
        "left"
    )
    .join(
        date_lookup,
        silver_loads.load_date == date_lookup.full_date,
        "left"
    )
    .select(
        "load_id",
        "date_key",
        "customer_key",
        "route_key",
        "load_date",
        "load_type",
        "weight_lbs",
        "pieces",
        "revenue",
        "fuel_surcharge",
        "accessorial_charges",
        "load_status",
        "booking_type"
    )
)

print(f"Load fact records: {load_fact.count():,}")
display(load_fact.limit(10))

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 20, Finished, Available, Finished, False)

Load fact records: 85,410


SynapseWidget(Synapse.DataFrame, 7dd36c51-5ff1-4589-a96d-7752852ea199)

In [19]:
duplicate_loads = (
    load_fact
    .groupBy("load_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate load IDs: {duplicate_loads}")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 21, Finished, Available, Finished, False)

Duplicate load IDs: 0


In [20]:
load_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_load")

print("gold_fact_load created successfully.")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 22, Finished, Available, Finished, False)

gold_fact_load created successfully.


In [21]:
driver_lookup = spark.table("gold_dim_driver").select(
    "driver_key",
    "driver_id"
)

truck_lookup = spark.table("gold_dim_truck").select(
    "truck_key",
    "truck_id"
)

trailer_lookup = spark.table("gold_dim_trailer").select(
    "trailer_key",
    "trailer_id"
)

trip_fact = (
    silver_trips
    .join(driver_lookup, "driver_id", "left")
    .join(truck_lookup, "truck_id", "left")
    .join(trailer_lookup, "trailer_id", "left")
    .join(
        date_lookup,
        silver_trips.dispatch_date == date_lookup.full_date,
        "left"
    )
    .select(
        "trip_id",
        "load_id",
        "date_key",
        "driver_key",
        "truck_key",
        "trailer_key",
        "dispatch_date",
        "actual_distance_miles",
        "actual_duration_hours",
        "fuel_gallons_used",
        "average_mpg",
        "idle_time_hours",
        "trip_status"
    )
)

print(f"Trip fact records: {trip_fact.count():,}")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 23, Finished, Available, Finished, False)

Trip fact records: 85,410


In [22]:
print(
    "Duplicate trip IDs:",
    trip_fact.groupBy("trip_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 24, Finished, Available, Finished, False)

Duplicate trip IDs: 0


In [23]:
trip_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_trip")

print("gold_fact_trip created successfully.")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 25, Finished, Available, Finished, False)

gold_fact_trip created successfully.


In [24]:
fuel_fact = (
    silver_fuel_purchases
    .join(driver_lookup, "driver_id", "left")
    .join(truck_lookup, "truck_id", "left")
    .join(
        date_lookup,
        silver_fuel_purchases.purchase_date == date_lookup.full_date,
        "left"
    )
    .select(
        "fuel_purchase_id",
        "trip_id",
        "date_key",
        "truck_key",
        "driver_key",
        "purchase_date",
        "location_city",
        "location_state",
        "gallons",
        "price_per_gallon",
        "total_cost",
        "fuel_card_number"
    )
)

print(f"Fuel fact records: {fuel_fact.count():,}")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 26, Finished, Available, Finished, False)

Fuel fact records: 196,442


In [25]:
fuel_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_fuel_purchase")

print("gold_fact_fuel_purchase created successfully.")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 27, Finished, Available, Finished, False)

gold_fact_fuel_purchase created successfully.


In [26]:
maintenance_fact = (
    silver_maintenance
    .join(
        truck_lookup,
        "truck_id",
        "left"
    )
    .join(
        date_lookup,
        silver_maintenance.maintenance_date == date_lookup.full_date,
        "left"
    )
    .select(
        "maintenance_id",
        "date_key",
        "truck_key",
        "truck_id",
        "maintenance_date",
        "maintenance_type",
        "odometer_reading",
        "labor_hours",
        "labor_cost",
        "parts_cost",
        "total_cost",
        "facility_location",
        "downtime_hours",
        "service_description"
    )
)

print(f"Maintenance fact records: {maintenance_fact.count():,}")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 28, Finished, Available, Finished, False)

Maintenance fact records: 2,920


In [27]:
maintenance_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_maintenance")

print("gold_fact_maintenance created successfully.")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 29, Finished, Available, Finished, False)

gold_fact_maintenance created successfully.


In [28]:
delivery_fact = (
    silver_delivery_events
    .join(
        date_lookup,
        F.to_date(silver_delivery_events.scheduled_datetime)
        == date_lookup.full_date,
        "left"
    )
    .join(
        spark.table("gold_dim_facility").select(
            "facility_key",
            "facility_id"
        ),
        "facility_id",
        "left"
    )
    .select(
        "event_id",
        "load_id",
        "trip_id",
        "date_key",
        "facility_key",
        "event_type",
        "scheduled_datetime",
        "actual_datetime",
        "detention_minutes",
        "on_time_flag",
        "location_city",
        "location_state"
    )
)

print(f"Delivery event fact records: {delivery_fact.count():,}")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 30, Finished, Available, Finished, False)

Delivery event fact records: 170,820


In [29]:
delivery_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_delivery_event")

print("gold_fact_delivery_event created successfully.")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 31, Finished, Available, Finished, False)

gold_fact_delivery_event created successfully.


In [1]:
safety_fact = (
    silver_safety_incidents
    .join(driver_lookup, "driver_id", "left")
    .join(truck_lookup, "truck_id", "left")
    .join(
        date_lookup,
        silver_safety_incidents.incident_date == date_lookup.full_date,
        "left"
    )
    .select(
        "incident_id",
        "trip_id",
        "date_key",
        "driver_key",
        "truck_key",
        "incident_date",
        "incident_type",
        "location_city",
        "location_state",
        "at_fault_flag",
        "injury_flag",
        "vehicle_damage_cost",
        "cargo_damage_cost",
        "claim_amount",
        "preventable_flag",
        "description"
    )
)

print(f"Safety incident fact records: {safety_fact.count():,}")

StatementMeta(, 4cbb4136-ee8d-439c-bb1c-289cc076b3ac, 3, Finished, Available, Finished, False)

NameError: name 'silver_safety_incidents' is not defined

In [31]:
safety_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_safety_incident")

print("gold_fact_safety_incident created successfully.")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 33, Finished, Available, Finished, False)

gold_fact_safety_incident created successfully.


In [32]:
gold_tables = [
    "gold_dim_date",
    "gold_dim_customer",
    "gold_dim_driver",
    "gold_dim_truck",
    "gold_dim_trailer",
    "gold_dim_facility",
    "gold_dim_route",
    "gold_fact_load",
    "gold_fact_trip",
    "gold_fact_fuel_purchase",
    "gold_fact_maintenance",
    "gold_fact_delivery_event",
    "gold_fact_safety_incident"
]

print("========== GOLD TABLE SUMMARY ==========")

for table_name in gold_tables:
    count = spark.table(table_name).count()
    print(f"{table_name}: {count:,}")

print("========================================")

StatementMeta(, bdd7622a-d05a-4fd4-a581-3f97b663109e, 34, Finished, Available, Finished, False)

========== GOLD TABLE SUMMARY ==========
gold_dim_date: 4,724
gold_dim_customer: 200
gold_dim_driver: 150
gold_dim_truck: 120
gold_dim_trailer: 180
gold_dim_facility: 50
gold_dim_route: 58
gold_fact_load: 85,410
gold_fact_trip: 85,410
gold_fact_fuel_purchase: 196,442
gold_fact_maintenance: 2,920
gold_fact_delivery_event: 170,820
gold_fact_safety_incident: 170


In [4]:
from pyspark.sql import functions as F
print("========== GOLD STAR SCHEMA VALIDATION ==========")

# Dimensions
dim_customer = spark.table("gold_dim_customer")
dim_driver = spark.table("gold_dim_driver")
dim_truck = spark.table("gold_dim_truck")
dim_trailer = spark.table("gold_dim_trailer")
dim_facility = spark.table("gold_dim_facility")
dim_route = spark.table("gold_dim_route")
dim_date = spark.table("gold_dim_date")

# Facts
fact_load = spark.table("gold_fact_load")
fact_trip = spark.table("gold_fact_trip")
fact_fuel = spark.table("gold_fact_fuel_purchase")
fact_maintenance = spark.table("gold_fact_maintenance")
fact_delivery = spark.table("gold_fact_delivery_event")
fact_safety = spark.table("gold_fact_safety_incident")


def check_fk(fact_df, fact_key, dim_df, dim_key):
    return (
        fact_df
        .filter(F.col(fact_key).isNotNull())
        .join(
            dim_df.select(dim_key),
            fact_df[fact_key] == dim_df[dim_key],
            "left_anti"
        )
        .count()
    )


# Load relationships
invalid_load_customer = check_fk(
    fact_load, "customer_key", dim_customer, "customer_key"
)

invalid_load_route = check_fk(
    fact_load, "route_key", dim_route, "route_key"
)

invalid_load_date = check_fk(
    fact_load, "date_key", dim_date, "date_key"
)


# Trip relationships
invalid_trip_driver = check_fk(
    fact_trip, "driver_key", dim_driver, "driver_key"
)

invalid_trip_truck = check_fk(
    fact_trip, "truck_key", dim_truck, "truck_key"
)

invalid_trip_trailer = check_fk(
    fact_trip, "trailer_key", dim_trailer, "trailer_key"
)

invalid_trip_date = check_fk(
    fact_trip, "date_key", dim_date, "date_key"
)


# Fuel relationships
invalid_fuel_driver = check_fk(
    fact_fuel, "driver_key", dim_driver, "driver_key"
)

invalid_fuel_truck = check_fk(
    fact_fuel, "truck_key", dim_truck, "truck_key"
)

invalid_fuel_date = check_fk(
    fact_fuel, "date_key", dim_date, "date_key"
)


# Maintenance relationships
invalid_maintenance_truck = check_fk(
    fact_maintenance, "truck_key", dim_truck, "truck_key"
)

invalid_maintenance_date = check_fk(
    fact_maintenance, "date_key", dim_date, "date_key"
)


# Delivery relationships
invalid_delivery_facility = check_fk(
    fact_delivery, "facility_key", dim_facility, "facility_key"
)

invalid_delivery_date = check_fk(
    fact_delivery, "date_key", dim_date, "date_key"
)


# Safety relationships
invalid_safety_driver = check_fk(
    fact_safety, "driver_key", dim_driver, "driver_key"
)

invalid_safety_truck = check_fk(
    fact_safety, "truck_key", dim_truck, "truck_key"
)

invalid_safety_date = check_fk(
    fact_safety, "date_key", dim_date, "date_key"
)


print("========== RESULTS ==========")

print(f"Invalid Load → Customer: {invalid_load_customer}")
print(f"Invalid Load → Route: {invalid_load_route}")
print(f"Invalid Load → Date: {invalid_load_date}")

print(f"Invalid Trip → Driver: {invalid_trip_driver}")
print(f"Invalid Trip → Truck: {invalid_trip_truck}")
print(f"Invalid Trip → Trailer: {invalid_trip_trailer}")
print(f"Invalid Trip → Date: {invalid_trip_date}")

print(f"Invalid Fuel → Driver: {invalid_fuel_driver}")
print(f"Invalid Fuel → Truck: {invalid_fuel_truck}")
print(f"Invalid Fuel → Date: {invalid_fuel_date}")

print(f"Invalid Maintenance → Truck: {invalid_maintenance_truck}")
print(f"Invalid Maintenance → Date: {invalid_maintenance_date}")

print(f"Invalid Delivery → Facility: {invalid_delivery_facility}")
print(f"Invalid Delivery → Date: {invalid_delivery_date}")

print(f"Invalid Safety → Driver: {invalid_safety_driver}")
print(f"Invalid Safety → Truck: {invalid_safety_truck}")
print(f"Invalid Safety → Date: {invalid_safety_date}")

print("================================")

StatementMeta(, 4cbb4136-ee8d-439c-bb1c-289cc076b3ac, 6, Finished, Available, Finished, False)

========== GOLD STAR SCHEMA VALIDATION ==========
========== RESULTS ==========
Invalid Load → Customer: 0
Invalid Load → Route: 0
Invalid Load → Date: 0
Invalid Trip → Driver: 0
Invalid Trip → Truck: 0
Invalid Trip → Trailer: 0
Invalid Trip → Date: 0
Invalid Fuel → Driver: 0
Invalid Fuel → Truck: 0
Invalid Fuel → Date: 0
Invalid Maintenance → Truck: 0
Invalid Maintenance → Date: 0
Invalid Delivery → Facility: 0
Invalid Delivery → Date: 0
Invalid Safety → Driver: 0
Invalid Safety → Truck: 0
Invalid Safety → Date: 0


In [5]:
from pyspark.sql import functions as F

print("========== GOLD BUSINESS KPI VALIDATION ==========")

# Load Gold facts
fact_load = spark.table("gold_fact_load")
fact_trip = spark.table("gold_fact_trip")
fact_fuel = spark.table("gold_fact_fuel_purchase")
fact_maintenance = spark.table("gold_fact_maintenance")
fact_delivery = spark.table("gold_fact_delivery_event")
fact_safety = spark.table("gold_fact_safety_incident")

# -----------------------------
# LOAD KPIs
# -----------------------------
load_kpis = fact_load.select(
    F.count("*").alias("load_count"),
    F.sum("revenue").alias("total_revenue"),
    F.sum("fuel_surcharge").alias("total_fuel_surcharge"),
    F.sum("accessorial_charges").alias("total_accessorial_charges"),
    F.sum("weight_lbs").alias("total_weight_lbs"),
    F.sum("pieces").alias("total_pieces")
).collect()[0]

# -----------------------------
# TRIP KPIs
# -----------------------------
trip_kpis = fact_trip.select(
    F.count("*").alias("trip_count"),
    F.sum("actual_distance_miles").alias("total_distance_miles"),
    F.sum("fuel_gallons_used").alias("total_fuel_gallons"),
    F.sum("actual_duration_hours").alias("total_duration_hours"),
    F.sum("idle_time_hours").alias("total_idle_hours")
).collect()[0]

# -----------------------------
# FUEL KPIs
# -----------------------------
fuel_kpis = fact_fuel.select(
    F.count("*").alias("fuel_purchase_count"),
    F.sum("gallons").alias("purchased_fuel_gallons"),
    F.sum("total_cost").alias("total_fuel_cost")
).collect()[0]

# -----------------------------
# MAINTENANCE KPIs
# -----------------------------
maintenance_kpis = fact_maintenance.select(
    F.count("*").alias("maintenance_count"),
    F.sum("total_cost").alias("total_maintenance_cost"),
    F.sum("downtime_hours").alias("total_downtime_hours")
).collect()[0]

# -----------------------------
# SAFETY KPIs
# -----------------------------
safety_kpis = fact_safety.select(
    F.count("*").alias("incident_count"),
    F.sum("claim_amount").alias("total_claim_amount"),
    F.sum("vehicle_damage_cost").alias("vehicle_damage_cost"),
    F.sum("cargo_damage_cost").alias("cargo_damage_cost")
).collect()[0]

# -----------------------------
# PRINT RESULTS
# -----------------------------
print("\n----- LOAD KPIs -----")
print(f"Load Count: {load_kpis['load_count']}")
print(f"Total Revenue: {load_kpis['total_revenue']}")
print(f"Fuel Surcharge: {load_kpis['total_fuel_surcharge']}")
print(f"Accessorial Charges: {load_kpis['total_accessorial_charges']}")
print(f"Total Weight (lbs): {load_kpis['total_weight_lbs']}")
print(f"Total Pieces: {load_kpis['total_pieces']}")

print("\n----- TRIP KPIs -----")
print(f"Trip Count: {trip_kpis['trip_count']}")
print(f"Total Distance (miles): {trip_kpis['total_distance_miles']}")
print(f"Fuel Used (gallons): {trip_kpis['total_fuel_gallons']}")
print(f"Duration (hours): {trip_kpis['total_duration_hours']}")
print(f"Idle Time (hours): {trip_kpis['total_idle_hours']}")

print("\n----- FUEL KPIs -----")
print(f"Fuel Purchase Count: {fuel_kpis['fuel_purchase_count']}")
print(f"Purchased Fuel (gallons): {fuel_kpis['purchased_fuel_gallons']}")
print(f"Total Fuel Cost: {fuel_kpis['total_fuel_cost']}")

print("\n----- MAINTENANCE KPIs -----")
print(f"Maintenance Count: {maintenance_kpis['maintenance_count']}")
print(f"Total Maintenance Cost: {maintenance_kpis['total_maintenance_cost']}")
print(f"Total Downtime (hours): {maintenance_kpis['total_downtime_hours']}")

print("\n----- SAFETY KPIs -----")
print(f"Incident Count: {safety_kpis['incident_count']}")
print(f"Total Claims: {safety_kpis['total_claim_amount']}")
print(f"Vehicle Damage Cost: {safety_kpis['vehicle_damage_cost']}")
print(f"Cargo Damage Cost: {safety_kpis['cargo_damage_cost']}")

print("\n========== KPI VALIDATION COMPLETE ==========")

StatementMeta(, 4cbb4136-ee8d-439c-bb1c-289cc076b3ac, 7, Finished, Available, Finished, False)

========== GOLD BUSINESS KPI VALIDATION ==========

----- LOAD KPIs -----
Load Count: 85410
Total Revenue: 262525800.28999907
Fuel Surcharge: 29976528.65000156
Accessorial Charges: 6119100
Total Weight (lbs): 2346852874
Total Pieces: 1235928

----- TRIP KPIs -----
Trip Count: 85410
Total Distance (miles): 122159201
Fuel Used (gallons): 18946280.099999927
Duration (hours): 2136503.2999999775
Idle Time (hours): 598790.9999999997

----- FUEL KPIs -----
Fuel Purchase Count: 196442
Purchased Fuel (gallons): 24519037.799999997
Total Fuel Cost: 95592992.04000017

----- MAINTENANCE KPIs -----
Maintenance Count: 2920
Total Maintenance Cost: 5730573.280000002
Total Downtime (hours): 72230.50000000004

----- SAFETY KPIs -----
Incident Count: 170
Total Claims: 2653171.820000001
Vehicle Damage Cost: 1603561.1100000003
Cargo Damage Cost: 1049610.71

========== KPI VALIDATION COMPLETE ==========
